# Hippocampus MRI Segmentation with MONAI 🧠

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminesu/mri-hippocampus-segmentation/blob/main/notebook.ipynb)

An end-to-end **3D medical image segmentation** pipeline that segments the anterior and
posterior **hippocampus** from brain MRI, built with [**MONAI**](https://monai.io/) and PyTorch.

- **Data:** Medical Segmentation Decathlon — *Task04 Hippocampus* (auto-downloaded, ~27 MB, no login).
- **Model:** 3D U-Net (`monai.networks.nets.UNet`).
- **Loss / metric:** Dice + cross-entropy loss, mean Dice score (per foreground class).
- **Runtime:** trains in a few minutes on a free Colab GPU (`Runtime → Change runtime type → GPU`).

> Built as a portfolio project by Emine Şevval Eş Uzunay. Runs top-to-bottom on Colab with no setup.

## 1. Install dependencies

MONAI + NiBabel for reading NIfTI volumes. (PyTorch is pre-installed on Colab.)

In [ ]:
%pip install -q "monai>=1.3" nibabel matplotlib tqdm

## 2. Imports & configuration

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from monai.apps import DecathlonDataset
from monai.data import DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    NormalizeIntensityd, SpatialPadd, RandCropByPosNegLabeld,
    RandFlipd, RandRotate90d, EnsureTyped,
    Activations, AsDiscrete,
)

set_determinism(seed=42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else
                      ("mps" if torch.backends.mps.is_available() else "cpu"))
ROI = (32, 32, 32)          # patch size for training / sliding-window inference
NUM_CLASSES = 3             # background + anterior + posterior hippocampus
MAX_EPOCHS = 30             # bump this up once you confirm the pipeline runs
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)   # DecathlonDataset requires the root dir to exist
os.makedirs("assets", exist_ok=True)   # where result figures are saved

print("Device:", DEVICE)

## 3. Data transforms

Dictionary transforms operate on `{"image", "label"}` pairs. Training uses class-balanced random
patches plus light augmentation; validation keeps the full volume for sliding-window inference.

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    SpatialPadd(keys=["image", "label"], spatial_size=ROI),
    RandCropByPosNegLabeld(
        keys=["image", "label"], label_key="label", spatial_size=ROI,
        pos=1, neg=1, num_samples=4, image_key="image", image_threshold=0,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandRotate90d(keys=["image", "label"], prob=0.5, max_k=3),
    EnsureTyped(keys=["image", "label"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
    EnsureTyped(keys=["image", "label"]),
])

## 4. Download the dataset

`DecathlonDataset` downloads and caches *Task04_Hippocampus* on first run, then serves the
transformed volumes. The task ships a training set with labels, which we split into train/val.

In [ ]:
train_ds = DecathlonDataset(
    root_dir=DATA_DIR, task="Task04_Hippocampus", section="training",
    transform=train_transforms, download=True, cache_rate=1.0,
    val_frac=0.2, num_workers=2,
)
val_ds = DecathlonDataset(
    root_dir=DATA_DIR, task="Task04_Hippocampus", section="validation",
    transform=val_transforms, download=False, cache_rate=1.0,
    val_frac=0.2, num_workers=2,
)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=2)

print(f"Training volumes:   {len(train_ds)}")
print(f"Validation volumes: {len(val_ds)}")

## 5. Peek at one volume

Middle axial slice of an MRI with its ground-truth label overlaid (2 hippocampus sub-regions).

In [ ]:
check = val_ds[0]
img, lbl = check["image"], check["label"]
z = img.shape[-1] // 2

fig, ax = plt.subplots(1, 3, figsize=(11, 4))
ax[0].imshow(img[0, :, :, z], cmap="gray");            ax[0].set_title("MRI")
ax[1].imshow(lbl[0, :, :, z]);                          ax[1].set_title("Ground-truth label")
ax[2].imshow(img[0, :, :, z], cmap="gray")
ax[2].imshow(np.ma.masked_where(lbl[0, :, :, z] == 0, lbl[0, :, :, z]), alpha=0.6)
ax[2].set_title("Overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

## 6. Model, loss, optimizer, metric

In [ ]:
model = UNet(
    spatial_dims=3, in_channels=1, out_channels=NUM_CLASSES,
    channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2),
    num_res_units=2, norm="batch",
).to(DEVICE)

loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
dice_metric = DiceMetric(include_background=False, reduction="mean")

post_pred = Compose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=NUM_CLASSES)])
post_label = Compose([AsDiscrete(to_onehot=NUM_CLASSES)])

n_params = sum(p.numel() for p in model.parameters())
print(f"3D U-Net with {n_params/1e6:.2f}M parameters on {DEVICE}")

## 7. Training loop

Each epoch trains on random patches, then evaluates mean Dice on the validation volumes using
sliding-window inference. The best-scoring checkpoint is saved to `best_model.pth`.

In [ ]:
train_losses, val_dices = [], []
best_dice = -1.0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        x = batch["image"].to(DEVICE)
        y = batch["label"].to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)

    # ---- validation ----
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            x = batch["image"].to(DEVICE)
            y = batch["label"].to(DEVICE)
            logits = sliding_window_inference(x, ROI, 4, model)
            preds = [post_pred(p) for p in decollate_batch(logits)]
            labels = [post_label(l) for l in decollate_batch(y)]
            dice_metric(y_pred=preds, y=labels)
        mean_dice = dice_metric.aggregate().item()
        dice_metric.reset()
    val_dices.append(mean_dice)

    if mean_dice > best_dice:
        best_dice = mean_dice
        torch.save(model.state_dict(), "best_model.pth")
        star = "  <- best"
    else:
        star = ""
    print(f"epoch {epoch:02d}/{MAX_EPOCHS}  loss={epoch_loss:.4f}  val_dice={mean_dice:.4f}{star}")

print(f"\nBest validation Dice: {best_dice:.4f}")

## 8. Training curves

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(range(1, len(train_losses) + 1), train_losses, marker="o")
ax[0].set_title("Training loss"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("DiceCE loss")
ax[1].plot(range(1, len(val_dices) + 1), val_dices, marker="o", color="green")
ax[1].set_title("Validation mean Dice"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("Dice")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("assets/training_curves.png", dpi=120, bbox_inches="tight"); plt.show()

## 9. Qualitative results

Ground-truth vs. predicted segmentation on a validation volume (best checkpoint).

In [ ]:
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

sample = val_ds[0]
x = sample["image"].unsqueeze(0).to(DEVICE)
y = sample["label"]
with torch.no_grad():
    logits = sliding_window_inference(x, ROI, 4, model)
pred = torch.argmax(logits, dim=1).cpu()[0]

img = sample["image"][0]
gt = y[0]
# choose the slice with the most labelled voxels
z = int(gt.sum(dim=(0, 1)).argmax())

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img[:, :, z], cmap="gray"); ax[0].set_title("MRI"); ax[0].axis("off")
ax[1].imshow(img[:, :, z], cmap="gray")
ax[1].imshow(np.ma.masked_where(gt[:, :, z] == 0, gt[:, :, z]), alpha=0.6)
ax[1].set_title("Ground truth"); ax[1].axis("off")
ax[2].imshow(img[:, :, z], cmap="gray")
ax[2].imshow(np.ma.masked_where(pred[:, :, z] == 0, pred[:, :, z]), alpha=0.6)
ax[2].set_title("Prediction"); ax[2].axis("off")
plt.tight_layout(); plt.savefig("assets/prediction.png", dpi=120, bbox_inches="tight"); plt.show()

## 10. Notes & next steps

- **Increase `MAX_EPOCHS`** (e.g. 100–200) for a stronger Dice score — 30 is just a fast sanity run.
- Report **per-class Dice** (anterior vs. posterior hippocampus) with `reduction="mean_batch"`.
- Add **k-fold cross-validation** and test-time augmentation for a more honest estimate.
- Swap the backbone for a `SegResNet` or a pretrained encoder to compare architectures.

---
*References: [MONAI](https://monai.io/) · [Medical Segmentation Decathlon](http://medicaldecathlon.com/).*